In [ ]:
# Install required libraries
!pip install -U transformers accelerate huggingface_hub sentencepiece firebase-admin
# -------------------------------
# Imports
# -------------------------------
import os
import torch
import firebase_admin
from firebase_admin import credentials, firestore
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Libraries loaded")

# -------------------------------
# Debug Kaggle input path
# -------------------------------
print("Available Kaggle input files:")

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

# -------------------------------
# Initialize Firebase
# -------------------------------
cred = credentials.Certificate(
    "/kaggle/input/firebase-key/firebase_key.json"
)

firebase_admin.initialize_app(cred)

db = firestore.client()

print("Connected to Firestore")

# -------------------------------
# Fix HuggingFace download issues
# -------------------------------
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

# -------------------------------
# Load TinyLlama
# -------------------------------
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading TinyLlama model...")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print("TinyLlama model loaded")

# -------------------------------
# Email generation function
# -------------------------------
def generate_emails(prompt, num=5):

    emails = set()
    attempts = 0

    while len(emails) < num and attempts < num * 8:

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.9,
            top_p=0.95
        )

        text = tokenizer.decode(output[0], skip_special_tokens=True)

        email = text.replace(prompt, "").strip()

        if len(email) > 100:
            emails.add(email)

        attempts += 1

    return list(emails)

# -------------------------------
# Fetch feedback emails
# -------------------------------
docs = db.collection("user_feedback") \
         .where("generation_status", "==", "pending") \
         .stream()

feedback_list = list(docs)

print("Pending feedback emails:", len(feedback_list))

# -------------------------------
# Generate synthetic emails
# -------------------------------
for doc in feedback_list:

    data = doc.to_dict()

    subject = data["subject"]
    content = data["content"]
    label = data["user_label"]

    prompt = f"""
You are generating synthetic training data for an email phishing detector.

Task:
Write ONE complete {label} email.

Requirements:
- Must include a Subject line
- Must include greeting
- Must include full email body
- Must include closing signature
- Do NOT copy the example
- Write a new email with similar intent

Example Email

Subject: {subject}

Content:
{content}

Now generate a new similar {label} email.
"""

    generated_emails = generate_emails(prompt, num=5)

    print("Generated emails:", len(generated_emails))

    for email_text in generated_emails:

        lines = email_text.split("\n")

        subject_line = subject
        body = email_text

        if "Subject:" in lines[0]:
            subject_line = lines[0]
            body = "\n".join(lines[1:])

        db.collection("training_emails").add({

            "source_email_id": data["email_id"],
            "subject": subject_line,
            "content": body,
            "label": label,
            "training_status": "unused"

        })

    # Update status
    doc.reference.update({

        "generation_status": "completed"

    })

print("Synthetic emails generated and stored successfully.")